In [22]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



Finding Condensate-associated RNAs to put in affinity matrix
> [From RPS](https://rps.renlab.cn/#/Search)



In [23]:
df = pd.read_csv('/content/drive/My Drive/Modeling Project/RPS_HTP-perturbation_detail_table_0903.csv')
df.head()

/tmp/ipykernel_792/1518156408.py:1: DtypeWarning: Columns (19,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/My Drive/Modeling Project/RPS_HTP-perturbation_detail_table_0903.csv')


,rps_id,event_type,gene_symbol,organism,llps_id,pmid,cell_line_tissue,location,experiment_design,llps_detection_methods,...,db,gene_region,position,ref,control_mut_info,treatment_mut_info,control_mean,control_stderr,treatment_mean,treatment_stderr
0,RPS_137659,Expression,0610005C13Rik,Mus musculus,LLPS_L3_7,34088848|https://pubmed.ncbi.nlm.nih.gov/34088848,Brown adipose tissue,Cytoplasm,FAM195A KO vs Wild type,Immunofluorescence,...,NaN,NaN,NaN,NaN,NaN,NaN,2.7829,0.5299,4.3753,0.7947
1,RPS_133050,AS,1700022N22Rik,Mus musculus,LLPS_L3_7,34088848|https://pubmed.ncbi.nlm.nih.gov/34088848,Brown adipose tissue,Cytoplasm,FAM195A KO vs Wild type,Immunofluorescence,...,NaN,NaN,NaN,NaN,NaN,NaN,0.4443,0.2940,0.4443,0.2940
2,RPS_115529,AS,Ap5z1,Mus musculus,LLPS_L3_7,34088848|https://pubmed.ncbi.nlm.nih.gov/34088848,Brown adipose tissue,Cytoplasm,FAM195A KO vs Wild type,Immunofluorescence,...,NaN,NaN,NaN,NaN,NaN,NaN,0.2040,0.0150,0.0317,0.0317
3,RPS_25247,APA,PAPOLA,Homo sapiens,LLPS_L3_3,36354291|https://pubmed.ncbi.nlm.nih.gov/36354291,HEK293FRT cells,Nucleus,ARID1A KD vs Wild type,Conventional microscopy,...,NaN,NaN,NaN,NaN,NaN,NaN,0.9533,0.0088,0.9100,0.0200
4,RPS_25247,APA,PAPOLA,Homo sapiens,LLPS_L3_5,36354291|https://pubmed.ncbi.nlm.nih.gov/36354291,HEK293FRT cells,Nucleus,NEAT1 KD vs Wild type,Conventional microscopy,...,NaN,NaN,NaN,NaN,NaN,NaN,0.9967,0.0033,0.8167,0.0145


In [24]:
filtered_df = df[df['event_type'] == 'Expression']


In [25]:
filtered_df.head()

,rps_id,event_type,gene_symbol,organism,llps_id,pmid,cell_line_tissue,location,experiment_design,llps_detection_methods,...,db,gene_region,position,ref,control_mut_info,treatment_mut_info,control_mean,control_stderr,treatment_mean,treatment_stderr
0,RPS_137659,Expression,0610005C13Rik,Mus musculus,LLPS_L3_7,34088848|https://pubmed.ncbi.nlm.nih.gov/34088848,Brown adipose tissue,Cytoplasm,FAM195A KO vs Wild type,Immunofluorescence,...,NaN,NaN,NaN,NaN,NaN,NaN,2.7829,0.5299,4.3753,0.7947
15,RPS_20161,Expression,PAPPA,Homo sapiens,LLPS_L3_1,34901782|https://pubmed.ncbi.nlm.nih.gov/34901782,HEK293 cells,Cytoplasm,Sodium arsenite treated vs Control,"Immunofluorescence, Confocal microscopy",...,NaN,NaN,NaN,NaN,NaN,NaN,0.4060,0.4060,4.2024,2.7565
16,RPS_85683,Expression,PAPPA2,Homo sapiens,LLPS_L3_3,36354291|https://pubmed.ncbi.nlm.nih.gov/36354291,HEK293FRT cells,Nucleus,ARID1A KD vs Wild type,Conventional microscopy,...,NaN,NaN,NaN,NaN,NaN,NaN,1.5645,0.7873,3.5698,0.0983
17,RPS_85684,Expression,PAPPA2,Homo sapiens,LLPS_L3_1,34901782|https://pubmed.ncbi.nlm.nih.gov/34901782,HEK293 cells,Cytoplasm,Sodium arsenite treated vs Control,"Immunofluorescence, Confocal microscopy",...,NaN,NaN,NaN,NaN,NaN,NaN,1.8893,0.5608,3.7845,2.8875
29,RPS_37765,Expression,PAQR8,Homo sapiens,LLPS_L3_3,36354291|https://pubmed.ncbi.nlm.nih.gov/36354291,HEK293FRT cells,Nucleus,ARID1A KD vs Wild type,Conventional microscopy,...,NaN,NaN,NaN,NaN,NaN,NaN,7.8904,0.0592,9.3015,0.0926


In [26]:
unique_genes = df['gene_symbol'].drop_duplicates()


In [27]:
unique_genes

,gene_symbol
0,0610005C13Rik
1,1700022N22Rik
2,Ap5z1
3,PAPOLA
14,PAPOLG
...,...
117153,Ap5s1
117154,PANK4
117155,PANTR1
117157,PAOX


Finding RNA-protein associations from RNA in condensate network


> From [RNAffinityDB](https://www.ultrarnalab.com/RNAffinityDB.php?page=download)



In [28]:
rdb_df = pd.read_csv('/content/drive/My Drive/Modeling Project/RNAffinity_DB_kon_koff_v20261004.csv')
rdb_df.head()

,PMID,measure_method,protein_name,protein_source,protein_sequence,RNA_name,RNA_source,RNA_sequence,Affinity_mean(nM),Affinity_std(nM),association(M^-1s^-1),dissociation(s^-1)
0,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R49_deleteG59,synthetic,GGGAGACCGGAAUUCUGGGCGCAGUGUCAUUGACGCUGCGUACAGA...,non-binding,NaN,NaN,NaN
1,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R45_deleteU60,synthetic,GGGAGACCGGAAUUCUGGGCGCAGUGUCAUUGACGCUGCGGACAGA...,non-binding,NaN,NaN,NaN
2,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R46_deleteA61,synthetic,GGGAGACCGGAAUUCUGGGCGCAGUGUCAUUGACGCUGCGGUCAGA...,non-binding,NaN,NaN,NaN
3,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R34_deleteG35G36,synthetic,GGGAGACCGGAAUUCUGCGCAGUGUCAUUGACGCUGCGGUACAGAA...,non-binding,NaN,NaN,NaN
4,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R36_deleteG59-A61,synthetic,GGGAGACCGGAAUUCUGGGCGCAGUGUCAUUGACGCUGCGCAGAAU...,non-binding,NaN,NaN,NaN


In [29]:
rdb_df

,PMID,measure_method,protein_name,protein_source,protein_sequence,RNA_name,RNA_source,RNA_sequence,Affinity_mean(nM),Affinity_std(nM),association(M^-1s^-1),dissociation(s^-1)
0,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R49_deleteG59,synthetic,GGGAGACCGGAAUUCUGGGCGCAGUGUCAUUGACGCUGCGUACAGA...,non-binding,NaN,NaN,NaN
1,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R45_deleteU60,synthetic,GGGAGACCGGAAUUCUGGGCGCAGUGUCAUUGACGCUGCGGACAGA...,non-binding,NaN,NaN,NaN
2,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R46_deleteA61,synthetic,GGGAGACCGGAAUUCUGGGCGCAGUGUCAUUGACGCUGCGGUCAGA...,non-binding,NaN,NaN,NaN
3,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R34_deleteG35G36,synthetic,GGGAGACCGGAAUUCUGCGCAGUGUCAUUGACGCUGCGGUACAGAA...,non-binding,NaN,NaN,NaN
4,1871141,Filter-binding assay (filter binding),REV,Human immunodeficiency virus type 1 group M su...,MAGRSGDSDEDLLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,R36_deleteG59-A61,synthetic,GGGAGACCGGAAUUCUGGGCGCAGUGUCAUUGACGCUGCGCAGAAU...,non-binding,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
10011,30971701,Surface plasmon resonance (SPR),mStau2 dsRBD1-2,Mus musculus_Stau2_1-208_WT_Q8CJ67,MANPKEKTPVCLVNELARFHSIQPQYKLLNESGPAHSKMFSVQLSL...,SRS2 + 5,Mus musculus,UAUAUACAUGCACAUGCAAUUUGUGUAUGUGUAUGUAUAUA,130,30.0,NaN,NaN
10012,30971701,Surface plasmon resonance (SPR),mStau2 dsRBD3-4,Mus musculus_Stau2_200-373_WT_Q8CJ67,DDKDANKSEISLVFEIALKRNMPVSFEVIKESGPPHMKSFVTRVSV...,SRS2 + 5,Mus musculus,UAUAUACAUGCACAUGCAAUUUGUGUAUGUGUAUGUAUAUA,18,16.0,NaN,NaN
10013,30971701,Surface plasmon resonance (SPR),mStau2 wt FL,Mus musculus_Stau2_1-570_WT_Q8CJ67,MANPKEKTPVCLVNELARFHSIQPQYKLLNESGPAHSKMFSVQLSL...,SRS2 + 5,Mus musculus,UAUAUACAUGCACAUGCAAUUUGUGUAUGUGUAUGUAUAUA,10.6,5.0,NaN,NaN
10014,30971701,Surface plasmon resonance (SPR),mStau2 dsRBD1-4 F40A H169A,Mus musculus_Stau2_1-373_F40A-H169A_Q8CJ67,MANPKEKTPVCLVNELARFHSIQPQYKLLNESGPAHSKMASVQLSL...,SRS2 + 5,Mus musculus,UAUAUACAUGCACAUGCAAUUUGUGUAUGUGUAUGUAUAUA,32,13.0,NaN,NaN


In [30]:
rdb_RNA = rdb_df['RNA_name'].drop_duplicates()

In [31]:
rdb_RNA

,RNA_name
0,R49_deleteG59
1,R45_deleteU60
2,R46_deleteA61
3,R34_deleteG35G36
4,R36_deleteG59-A61
...,...
9973,11-1-M17
9974,11-1-M18
9975,11-1-M19
9978,SRS* RNA


In [32]:
overlap_df = pd.DataFrame(
    [(rna_name, gene) for rna_name in rdb_RNA for gene in unique_genes if gene in rna_name],
    columns=["RNAs in RNAffinityDB", "GenesInRPS"]
)

print(overlap_df)

                  RNAs in RNAffinityDB GenesInRPS
0     R37_G34toA_C35to_A_C36toA_C62toU         C3
1     R37_G34toA_C35to_A_C36toA_C62toU         C6
2                       R40_G34C62toCG         C6
3                       R56_C37G58toUA         C3
4                       R41_C37G58toGC         C3
...                                ...        ...
1305                  7-nt RNA GCGGGCG        GCG
1306                  7-nt RNA CAGCGUC         GC
1307                  7-nt RNA CAGCGUC        GCG
1308                  7-nt RNA CACACAC      ACACA
1309                           Gk-mRNA         Gk

[1310 rows x 2 columns]


In [33]:
overlap_df.to_csv('overlap_df.csv')
#

Manually cleaned the overlap DF to find true overlaps

In [34]:
cleaned_overlap_df = pd.read_csv('/content/drive/My Drive/Modeling Project/cleaned_overlap_df.csv')
unique_RNAs_fror_RNAffinity = cleaned_overlap_df['RNAs in RNAffinityDB'].drop_duplicates()
unique_RNAs_fror_RNAffinity



,RNAs in RNAffinityDB
0,MBNL1 / 6
1,ITGB4 / 31
2,ADMLmut
3,INSR / 10
4,LCAT / 4
...,...
778,Malat1_start
779,Clip34_UG6
780,Dap1
781,rRAND 46-mer random RNA oligomer


In [35]:
filtered_df = rdb_df[rdb_df["RNA_name"].isin(unique_RNAs_fror_RNAffinity)].copy()

In [36]:
filtered_df

,PMID,measure_method,protein_name,protein_source,protein_sequence,RNA_name,RNA_source,RNA_sequence,Affinity_mean(nM),Affinity_std(nM),association(M^-1s^-1),dissociation(s^-1)
364,18549497,Electrophoretic mobility shift assay (EMSA),U2AF65,Homo sapiens_U2AF2_full-length_WT_P26368,MSDFDEFERQLNENKQERDKENRHRKRSHSRSRSRDRKRRSRSRDR...,MBNL1 / 6,Homo sapiens,CAUGUGCUCGCUGCCUGCUAAUUAAG,100000,NaN,NaN,NaN
365,18549497,Electrophoretic mobility shift assay (EMSA),U2AF65,Homo sapiens_U2AF2_full-length_WT_P26368,MSDFDEFERQLNENKQERDKENRHRKRSHSRSRSRDRKRRSRSRDR...,ITGB4 / 31,Homo sapiens,CCCUGGCUCACUCCCCUGCCCUGCAG,52000,NaN,NaN,NaN
366,18549497,Electrophoretic mobility shift assay (EMSA),U2AF65,Homo sapiens_U2AF2_full-length_WT_P26368,MSDFDEFERQLNENKQERDKENRHRKRSHSRSRSRDRKRRSRSRDR...,ADMLmut,Homo sapiens,UUCGUGCUGACCCUGUCCCGUAUUAGUCCACAGCUGCA,15800,6300.00,NaN,NaN
367,18549497,Electrophoretic mobility shift assay (EMSA),U2AF65,Homo sapiens_U2AF2_full-length_WT_P26368,MSDFDEFERQLNENKQERDKENRHRKRSHSRSRSRDRKRRSRSRDR...,INSR / 10,Homo sapiens,CAAAGGCGUUGGUUUUGUUUCCACAG,8800,1500.00,NaN,NaN
369,18549497,Electrophoretic mobility shift assay (EMSA),U2AF65,Homo sapiens_U2AF2_full-length_WT_P26368,MSDFDEFERQLNENKQERDKENRHRKRSHSRSRSRDRKRRSRSRDR...,LCAT / 4,Homo sapiens,GCCCUGACCCCUUCCACCCGCUGCAG,1900,300.00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
9670,42165134,Isothermal titration calorimetry (ITC),DasR R48A,Streptomyces coelicolor_DasR_1-254_R48A_Q9K492,MSTDVSSAENEGGATVRTARVPKYYRLKKHLLDMTRTQTPGTPVPP...,Dap1,in vitro-transcribed SELEX-derived DasR-bindin...,GGCCACCCGAGAUGCCCAGUGCCCUAGUGGCC,>1000,NaN,NaN,NaN
9671,42165134,Isothermal titration calorimetry (ITC),DasR R58A,Streptomyces coelicolor_DasR_1-254_R58A_Q9K492,MSTDVSSAENEGGATVRTARVPKYYRLKKHLLDMTRTQTPGTPVPP...,Dap1,in vitro-transcribed SELEX-derived DasR-bindin...,GGCCACCCGAGAUGCCCAGUGCCCUAGUGGCC,>1000,NaN,NaN,NaN
9672,42165134,Isothermal titration calorimetry (ITC),DasR R62A,Streptomyces coelicolor_DasR_1-254_R62A_Q9K492,MSTDVSSAENEGGATVRTARVPKYYRLKKHLLDMTRTQTPGTPVPP...,Dap1,in vitro-transcribed SELEX-derived DasR-bindin...,GGCCACCCGAGAUGCCCAGUGCCCUAGUGGCC,>1000,NaN,NaN,NaN
9802,9020172,Electrophoretic mobility shift assay (EMSA),qTBP42 quadruplex telomere-binding protein 42,Rattus norvegicus_qTBP42 (CBF-A/hnRNP A/B)_ful...,MSDAAEEQPMETTGATENGHEAAPEGEAPVEPSPGAAAPATPAGSG...,rRAND 46-mer random RNA oligomer,synthetic random RNA oligomer,GCGCGGAAGCUUGGCUGCAGAAUAUUGCUAGCGGGAAUUCGGCGCG,6.8,3.10,NaN,NaN


In [37]:
columns_to_keep = ["PMID", "protein_name", "RNA_name", "Affinity_mean(nM)", "Affinity_std(nM)"]

pared_filtered_df = filtered_df[columns_to_keep].copy()

In [38]:
pared_filtered_df

,PMID,protein_name,RNA_name,Affinity_mean(nM),Affinity_std(nM)
364,18549497,U2AF65,MBNL1 / 6,100000,NaN
365,18549497,U2AF65,ITGB4 / 31,52000,NaN
366,18549497,U2AF65,ADMLmut,15800,6300.00
367,18549497,U2AF65,INSR / 10,8800,1500.00
369,18549497,U2AF65,LCAT / 4,1900,300.00
...,...,...,...,...,...
9670,42165134,DasR R48A,Dap1,>1000,NaN
9671,42165134,DasR R58A,Dap1,>1000,NaN
9672,42165134,DasR R62A,Dap1,>1000,NaN
9802,9020172,qTBP42 quadruplex telomere-binding protein 42,rRAND 46-mer random RNA oligomer,6.8,3.10


In [40]:
#add RNA likelihoods of binding
probabilities = pared_filtered_df[["PMID", "protein_name", "RNA_name", "Affinity_mean(nM)"]].copy()

probabilities["Affinity_mean(nM)"] = (
    probabilities["Affinity_mean(nM)"]
    .astype(str)
    .str.replace(">", "", regex=False)
)

probabilities["Affinity_mean(nM)"] = pd.to_numeric(
    probabilities["Affinity_mean(nM)"], errors="coerce"
)



In [46]:
def calculate_pij(affinity, C=1):
  Kd = affinity / 1000
  cAB = (2*C+Kd - np.sqrt(Kd**2 + 4*C*Kd))/2
  pij = cAB/C
  return pij

In [47]:
probabilities["Probability"] = probabilities["Affinity_mean(nM)"].apply(calculate_pij)

In [48]:
probabilities

,PMID,protein_name,RNA_name,Affinity_mean(nM),Probability
364,18549497,U2AF65,MBNL1 / 6,100000.0,0.009805
365,18549497,U2AF65,ITGB4 / 31,52000.0,0.018525
366,18549497,U2AF65,ADMLmut,15800.0,0.056358
367,18549497,U2AF65,INSR / 10,8800.0,0.093400
369,18549497,U2AF65,LCAT / 4,1900.0,0.275933
...,...,...,...,...,...
9670,42165134,DasR R48A,Dap1,1000.0,0.381966
9671,42165134,DasR R58A,Dap1,1000.0,0.381966
9672,42165134,DasR R62A,Dap1,1000.0,0.381966
9802,9020172,qTBP42 quadruplex telomere-binding protein 42,rRAND 46-mer random RNA oligomer,6.8,0.920868
